In [1]:
!pip install openslide-python


In [2]:
!sudo apt-get install openslide-tools

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libopenslide0
Suggested packages:
  libtiff-tools
The following NEW packages will be installed:
  libopenslide0 openslide-tools
0 upgraded, 2 newly installed, 0 to remove and 2 not upgraded.
Need to get 104 kB of archives.
After this operation, 297 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libopenslide0 amd64 3.4.1+dfsg-5build1 [89.8 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 openslide-tools amd64 3.4.1+dfsg-5build1 [13.8 kB]
Fetched 104 kB in 6s (18.6 kB/s)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 2.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readli

In [4]:
#!sudo apt-get install python-openslide

#**Run Packages**

In [5]:
import os #operating system dependent functionality.
import re #regular expression
import sys #System
import cv2 #OpenCv
import PIL #Pillow
import glob #Global variable
import math
import datetime
import openslide #For reading WSIs
import numpy as np # Numpy
import pandas as pd #Pandas
from PIL import Image
import multiprocessing
import matplotlib.pyplot as plt
from openslide import OpenSlideError
from PIL import Image, ImageDraw, ImageFont

#**Mount Google Drive**

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
data_path = "/content/drive/MyDrive/Georgetown_University-Workshop/Pathology_Workshop/WSI_2_Patches/Pathology_data_raw_file"
total_files = os.listdir(data_path)
total_files

['test-001.svs', 'TUPAC-TE-226.svs', 'TCGA-49-4512.svs', 'TCGA-49-4514.svs']

In [8]:
# Arguments and variable values
SCALE_FACTOR = 32
patch_size=256 # 512, 70

In [ ]:
def filter_rgb_to_grayscale(np_img, output_type="uint8"):
  grayscale = np.dot(np_img[..., :3], [0.2125, 0.7154, 0.0721])
  if output_type != "float":
    grayscale = grayscale.astype("uint8")
  return grayscale


In [20]:
OUT_DIR = "/content/drive/MyDrive/Georgetown_University-Workshop/Pathology_Workshop/WSI_2_Patches/tiles"
os.makedirs(OUT_DIR, exist_ok=True)


In [21]:
TISSUE_THRESHOLD = 0.5  # 80% tissue required to save patch

In [22]:
def is_white_patch(patch, white_thresh=0.8, intensity_thresh=220):
    """
    Returns True if patch is mostly white/background
    """
    gray = cv2.cvtColor(np.array(patch), cv2.COLOR_RGB2GRAY)
    white_pixels = np.sum(gray > intensity_thresh)
    white_ratio = white_pixels / gray.size
    return white_ratio > white_thresh


In [23]:
def is_white_patch(patch, white_thresh=0.8, intensity_thresh=220):
    gray = cv2.cvtColor(np.array(patch), cv2.COLOR_RGB2GRAY)
    white_ratio = np.sum(gray > intensity_thresh) / gray.size
    return white_ratio > white_thresh


for file in total_files:
    if not file.endswith('.svs'):
        continue

    print(f"Processing: {file}")
    slide = openslide.open_slide(os.path.join(data_path, file))

    large_w, large_h = slide.dimensions
    level = slide.get_best_level_for_downsample(SCALE_FACTOR)

    # ---------- READ DOWNSAMPLED SLIDE ----------
    whole_slide = slide.read_region(
        (0, 0), level, slide.level_dimensions[level]
    ).convert("RGB")

    img = np.array(whole_slide)

    # ---------- TISSUE MASK CREATION ----------
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    _, s, _ = cv2.split(hsv)

    _, tissue_mask = cv2.threshold(
        s, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU
    )

    tissue_mask = cv2.medianBlur(tissue_mask, 7)

    # ---------- TISSUE CALCULATION ----------
    total_pixels = tissue_mask.size
    tissue_pixels = np.count_nonzero(tissue_mask)
    non_tissue_pixels = total_pixels - tissue_pixels

    tissue_pct = round((tissue_pixels / total_pixels) * 100, 2)
    non_tissue_pct = round((non_tissue_pixels / total_pixels) * 100, 2)

    print(f"Tissue: {tissue_pct}% | Non-tissue: {non_tissue_pct}%")

    # ---------- PATCH GRID ----------
    n_across = large_w // patch_size
    n_down = large_h // patch_size

    patch_id = 0

    for y in range(n_down):
        for x in range(n_across):

            # Map patch location to tissue mask coordinates
            mask_x = int((x * patch_size) / SCALE_FACTOR)
            mask_y = int((y * patch_size) / SCALE_FACTOR)
            mask_w = int(patch_size / SCALE_FACTOR)

            mask_patch = tissue_mask[
                mask_y:mask_y + mask_w,
                mask_x:mask_x + mask_w
            ]

            if mask_patch.size == 0:
                continue

            tissue_fraction = np.count_nonzero(mask_patch) / mask_patch.size

            # ---------- COARSE FILTER (MASK) ----------
            if tissue_fraction < TISSUE_THRESHOLD:
                continue

            # ---------- READ PATCH ----------
            patch = slide.read_region(
                (x * patch_size, y * patch_size),
                0,
                (patch_size, patch_size)
            ).convert("RGB")

            # ---------- FINE FILTER (WHITE CHECK) ----------
            if is_white_patch(patch):
                continue

            # ---------- SAVE PATCH ----------
            patch_name = f"{file}_patch_{patch_id}.png"
            patch.save(os.path.join(OUT_DIR, patch_name))
            patch_id += 1

    print(f"Saved {patch_id} tissue patches\n")


Processing: test-001.svs
Tissue: 18.31% | Non-tissue: 81.69%
Saved 0 tissue patches

Processing: TUPAC-TE-226.svs
Tissue: 57.28% | Non-tissue: 42.72%


KeyboardInterrupt: 